# Решения: Логирование и raise: контракт preprocessing

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Схема обязательных столбцов

In [ ]:
def require_columns(frame,required):
    missing=sorted(set(required)-set(frame.columns))
    if missing: raise KeyError(f"missing columns: {missing}")
    return True
assert require_columns(orders,{"order_id"}) is True


## Урок. 2. Уникальный ключ заказов

In [ ]:
def require_unique(frame,column):
    if frame[column].duplicated().any(): raise ValueError(f"duplicate {column}")
    return True
assert require_unique(orders,"order_id") is True


## Урок. 3. Неотрицательные оплаты

In [ ]:
def validate_payment_values(frame):
    if frame["payment_value"].isna().any(): raise ValueError("payment_value has missing values")
    if frame["payment_value"].lt(0).any(): raise ValueError("payment_value must be non-negative")
    return True
assert validate_payment_values(payments) is True


## Урок. 4. Обязательная дата покупки

In [ ]:
def validate_purchase_dates(frame):
    if frame["order_purchase_timestamp"].isna().any(): raise ValueError("order_purchase_timestamp has missing values")
    return True
assert validate_purchase_dates(orders) is True


## Урок. 5. Единый валидатор

In [ ]:
def validate_inputs(orders_df,customers_df,payments_df):
    require_columns(orders_df,{"order_id","customer_id","order_purchase_timestamp"})
    require_columns(customers_df,{"customer_id","customer_unique_id","customer_state"})
    require_columns(payments_df,{"order_id","payment_type","payment_value"})
    require_unique(orders_df,"order_id"); require_unique(customers_df,"customer_id"); require_unique(payments_df,"order_id")
    validate_purchase_dates(orders_df); validate_payment_values(payments_df); return True
assert validate_inputs(orders,customers,payments) is True


## Урок. 6. Лог успешного пути

In [ ]:
log=[]
validate_inputs(orders,customers,payments); log.append("validated input contracts")
merged=orders.merge(payments,on="order_id",validate="one_to_one"); log.append("merged orders and payments")
assert len(log)==2


## Урок. 7. Проверка плохого пути

In [ ]:
bad=payments.head(3).copy(); bad.loc[bad.index[0],"payment_value"]=-5
bad_log=[]; caught=""
try:
    validate_inputs(orders,customers,bad); bad_log.append("validated")
except ValueError as e: caught=str(e)
assert caught and bad_log==[]


## Урок. 8. Инженерная записка

In [ ]:
CONTRACT_NOTE="Assert в учебном ноутбуке проверяет ожидаемый результат задачи и быстро показывает нарушение. Raise внутри pipeline является частью публичного контракта: останавливает обработку плохого входа с понятным сообщением. Лог не заменяет проверку; он фиксирует только завершённые шаги и помогает найти границу сбоя."
assert len(CONTRACT_NOTE)>=280


## ДЗ. 1. Валидатор payments

In [ ]:
def validate_payments(frame):
    require_columns(frame,{"order_id","payment_type","payment_value"}); require_unique(frame,"order_id"); validate_payment_values(frame); return True
assert validate_payments(payments) is True


## ДЗ. 2. Три негативных теста

In [ ]:
messages=[]
cases=[payments.drop(columns="payment_type"),pd.concat([payments.head(1),payments.head(1)]),payments.head(2).assign(payment_value=[-1,2])]
for case in cases:
    try: validate_payments(case)
    except (KeyError,ValueError) as e: messages.append(str(e))
assert len(messages)==3


## ДЗ. 3. Структурированный лог

In [ ]:
audit=[{"step":"load","rows":len(orders),"columns":len(orders.columns)}]
validate_inputs(orders,customers,payments); audit.append({"step":"validate","rows":len(orders),"columns":len(orders.columns)})
x=orders.merge(payments,on="order_id"); audit.append({"step":"merge","rows":len(x),"columns":len(x.columns)})
assert len(audit)==3


## ДЗ. 4. Challenge: контракт как функция

In [ ]:
def prepare_inputs(o,c,p):
    copies=(o.copy(),c.copy(),p.copy()); validate_inputs(*copies)
    return copies,["copied input frames","validated input contracts"]
(o2,c2,p2),audit=prepare_inputs(orders,customers,payments)


## ДЗ. 5. Challenge: политика ошибок

In [ ]:
ERROR_NOTE="Политика fail-fast останавливает pipeline до join и агрегации, чтобы дефект не распространился в признаки. Сообщение должно назвать нарушенный столбец, правило и по возможности число или пример строк. Оно не должно молча исправлять вход: решение об удалении, заполнении или возврате источнику принимает владелец данных."
assert len(ERROR_NOTE)>=260
